# 03 - Análise exploratória (EDA)

Notebook responsável pela análise exploratória dos dados limpos em
`data/processed/`: estatísticas descritivas, correlações entre variáveis
climáticas e produção cafeeira, e geração de gráficos exploratórios
(salvos em `figuras/`).

Usa as funções de `src/risco.py`.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/processed/lavras_clima_limpo.csv")
df.shape

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.describe()

## Observações iniciais

1. **Temperatura mínima tem uma cauda fria acentuada**: a média da mínima
   diária é ~15,5 °C (plausível para Lavras, que fica a ~900 m de
   altitude), mas o valor mínimo absoluto é 3,1 °C — indício de eventos
   de geada, relevantes para risco na cafeicultura.
2. **Chuva é muito assimétrica (skewed)**: a mediana de precipitação
   diária é de apenas 0,1 mm, bem abaixo da média de 3,2 mm, e o máximo
   chega a 90,6 mm em um único dia. Isso mostra que a maior parte dos
   dias é seca, com poucos eventos de chuva intensa puxando a média
   para cima — típico do regime de chuvas concentradas em Lavras.

## Índices de risco climático

Usando as funções de `src/risco.py`, calculamos indicadores de risco relevantes para a cafeicultura: dias de geada (leve/severa), dias de chuva excessiva e ocorrências de veranico (sequência de dias secos dentro da estação chuvosa, out-mar).

In [ ]:
import sys
sys.path.append("..")

from src.risco import resumo_anual_risco, contar_sequencias_secas

df["time"] = pd.to_datetime(df["time"])

In [ ]:
resumo = resumo_anual_risco(df)
resumo

**Nota sobre geada**: com o limiar de geada leve (≤3°C), nenhum dia da série é classificado — a mínima absoluta observada na base é 3,1°C. Isso sugere uma limitação da fonte Open-Meteo (dado de modelo/reanálise em grade, não estação terrestre local), que tende a suavizar extremos. Para detectar geadas de fato, provavelmente será necessário complementar com dados de estação (ex.: INMET) em uma iteração futura.

In [ ]:
import matplotlib.pyplot as plt

resumo["chuva_excessiva"].plot(marker="o", figsize=(10, 4), title="Dias de chuva excessiva por ano em Lavras-MG")
plt.ylabel("Número de dias")
plt.xlabel("Ano")
plt.tight_layout()
plt.savefig("../figuras/chuva_excessiva_por_ano.png", dpi=150)
plt.show()

### Veranicos

Sequências de pelo menos 10 dias secos consecutivos ocorrendo dentro da estação chuvosa (outubro a março), quando a planta espera chuva regular.

In [ ]:
veranicos = contar_sequencias_secas(df)
print("Veranicos detectados:", len(veranicos))
veranicos